In [6]:
from langchain_community.vectorstores import  FAISS
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import  init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain



In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [8]:
loader = TextLoader("Langchain_Rag.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 300, chunk_overlap = 50)
chunks = splitter.split_documents(raw_docs)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}:")
    print(chunk.page_content)
    print("-" * 50)

vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"))

Chunk 1:
LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).
LangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.
--------------------------------------------------
Chunk 2:
The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.
LangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.
--------------------------------------------------
Chunk 3:
Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.
Agents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.
--------------------------------------------------
Chunk 4:
BM25 and vector-based retrieval can be combined in LangChain to su

/var/folders/ly/dps_wwpj7972w23hbq80tr9r0000gn/T/ipykernel_12786/2039970407.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  vectorstore = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2"))
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6964.59it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":3})

In [18]:
prompt = PromptTemplate.from_template(
    """You are a helpful assistant. You answer the user's question based on the following retrieved documents. If you don't know the answer, say you don't know.

    Context: {context}

    Question: {input}

    """)

llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)

In [19]:
document_chain  = create_stuff_documents_chain(llm = llm, prompt = prompt)
rag_chain = create_retrieval_chain(retriever=retriever,combine_docs_chain= document_chain)

In [20]:
query = {"input": "How does langchain support agent and Memory"}
response = rag_chain.invoke(query)

print(response["answer"])

LangChain builds two complementary pieces into its LLM‑centric workflow:

### Agents  
* **Tool‑use capability** – An agent is an LLM that can decide, from the user’s prompt, which external tool to invoke (e.g., a calculator, a search API, a custom function, or any other API/database).  
* **Dynamic planning** – The LLM not only picks a tool but also determines the order of calls, allowing it to chain several operations to solve a task.  
* **Extensibility** – You can plug in any callable (HTTP endpoint, database client, custom Python function, etc.) and expose it to the agent via a simple “tool description” so the model knows how and when to use it.

### Memory  
* **ConversationBufferMemory** – Stores the raw sequence of user‑assistant turns. Each new turn is appended, and the full buffer can be supplied back to the LLM so it “remembers” the exact dialogue history.  
* **ConversationSummaryMemory** – Instead of keeping every token, this variant periodically summarizes the prior conve

In [21]:
response

{'input': 'How does langchain support agent and Memory',
 'context': [Document(id='cf8886c1-850e-4b85-8114-eefc78b9635e', metadata={'source': 'Langchain_Rag.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='1f101f7e-e6ba-4159-afa6-426cac84e062', metadata={'source': 'Langchain_Rag.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='6460ace0-4c3b-4dc8-83b0-cba5cc15ba81', metadata={'source': 'Langchain_Rag.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order during a task.\nLangChain supp